# WAF Operational Excellence
- Assessment Audit

This notebook assesses adherence to the Well-Architected Framework (WAF) Operational Excellence pillar using the **4 Key Topics** structure.

## Scoring Model (1-5 Scale per Topic)
| Score | Rating | Description |
|-------|--------|-------------|
| 5 | Excellent | Best practices fully implemented, automated, monitored |
| 4 | Good | Most best practices implemented, minor gaps |
| 3 | Moderate | Basic implementation, significant improvement opportunities |
| 2 | Limited | Minimal implementation, major gaps |
| 1 | Poor | Not implemented or fundamentally flawed |

## 4 Key Topics Assessed

1. **Prepare**
- Data Modeling, Naming Conventions, Environment Strategy, Capacity Planning

2. **Implement**
- Infrastructure as Code, CI/CD Pipelines, Version Control, Schema Migration

3. **Collaborate**
- Documentation, Standards, Cross-Team Communication

4. **Automate**
- Task Scheduling, Alerting, Self-Healing

## Setup: Account Context

**Choose ONE of the following cells based on your environment:**

| Environment | Cell to Run |
|-------------|-------------|
| **Snowsight Notebooks** | Run the `get_active_session()` cell |
| **Cortex Code Desktop** | Run the `Session.builder` cell |

In [100]:
SCHEMA = "IE"
ACCOUNT_ID = 100058
DATABASE = "SNOWHOUSE_IMPORT"

print(f"Target: {DATABASE}.{SCHEMA}")
print(f"Account ID: {ACCOUNT_ID}")

Target: SNOWHOUSE_IMPORT.IE
Account ID: 100058


In [101]:
from snowflake.snowpark import Session
import pandas as pd
import os

connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse"
if 'session' not in globals():
    session = Session.builder.config("connection_name", connection_name).create()

print(f"Connected via: {connection_name}")
print(f"Current user: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Current role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")
print(f"\nAnalyzing: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}")

Connected via: snowhouse
Current user: EDSCOTT
Current role: TECHNICAL_ACCOUNT_MANAGER

Analyzing: SNOWHOUSE_IMPORT.IE | Account ID: 100058


## Cache Data

Run this cell to create temporary tables with filtered data for analysis.

In [102]:
session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE cache_query_history AS
SELECT 
    j.JOB_ID as QUERY_ID, j.DESCRIPTION as QUERY_TEXT,
    UPPER(REGEXP_SUBSTR(TRIM(j.DESCRIPTION), '^[A-Za-z_]+')) as QUERY_TYPE,
    j.USER_NAME,
    CASE WHEN UPPER(j.USER_NAME) = 'SYSTEM' THEN 'System'
        WHEN u.TYPE = 1 THEN 'Human User'
        WHEN u.TYPE = 2 THEN 'Service (Compute Pool)'
        WHEN u.TYPE = 3 THEN 'Service Account'
        ELSE 'Legacy/Unknown' END AS USER_TYPE,
    j.WAREHOUSE_NAME, j.CREATED_ON as START_TIME,
    j.TOTAL_DURATION as TOTAL_ELAPSED_TIME, j.ERROR_CODE, j.SESSION_ID,
    s.CLIENT_APP_ID as CLIENT_APPLICATION_ID,
    CASE WHEN s.CLIENT_APP_ID ILIKE '%terraform%' THEN 'Terraform'
        WHEN s.CLIENT_APP_ID ILIKE '%snowflake-cli%' OR s.CLIENT_APP_ID ILIKE '%snowcli%' THEN 'Snowflake CLI'
        WHEN s.CLIENT_APP_ID ILIKE '%dbt%' THEN 'dbt'
        WHEN s.CLIENT_APP_ID ILIKE '%schemachange%' THEN 'Schemachange'
        WHEN s.CLIENT_APP_ID ILIKE '%python%' THEN 'Python SDK'
        WHEN s.CLIENT_APP_ID ILIKE '%jdbc%' THEN 'JDBC'
        WHEN s.CLIENT_APP_ID ILIKE '%odbc%' THEN 'ODBC'
        ELSE 'Other/Manual' END AS CLIENT_TOOL,
    CASE WHEN s.CLIENT_APP_ID ILIKE '%terraform%' OR s.CLIENT_APP_ID ILIKE '%snowflake-cli%'
        OR s.CLIENT_APP_ID ILIKE '%snowcli%' OR s.CLIENT_APP_ID ILIKE '%dbt%'
        OR s.CLIENT_APP_ID ILIKE '%schemachange%' THEN TRUE ELSE FALSE END AS IS_IAC
FROM {DATABASE}.{SCHEMA}.JOB_ETL_V j
LEFT JOIN {DATABASE}.{SCHEMA}.SESSION_ETL_V s ON j.SESSION_ID = s.ID AND s.ACCOUNT_ID = {ACCOUNT_ID}
LEFT JOIN {DATABASE}.{SCHEMA}.USER_ETL_V u ON j.USER_NAME = u.NAME AND u.ACCOUNT_ID = {ACCOUNT_ID} AND u.DELETED_ON IS NULL
WHERE j.ACCOUNT_ID = {ACCOUNT_ID} AND j.CREATED_ON > DATEADD(day, -30, CURRENT_TIMESTAMP())
""").collect()
print("Created: cache_query_history")

session.sql("""CREATE OR REPLACE TEMPORARY TABLE cache_ddl_operations AS
SELECT * FROM cache_query_history WHERE QUERY_TYPE IN ('CREATE', 'ALTER', 'DROP', 'GRANT', 'REVOKE', 'REPLACE')""").collect()
print("Created: cache_ddl_operations\nTemp tables ready.")

Created: cache_query_history
Created: cache_ddl_operations
Temp tables ready.


---
# Key Topic 1: PREPARE

**Set the foundation for operational success.**

| Sub-Topic | DDM Assessable | Assessment Method |
|-----------|----------------|-------------------|
| Data Modeling | Partial | TABLE_ETL_V + Manual Questions |
| Naming Conventions | Yes | DATABASE_ETL_V, WAREHOUSE_ETL_V |
| Environment Strategy | Yes | DATABASE_ETL_V patterns |
| Capacity Planning | Partial | RESOURCE_MONITOR_ETL_V + Manual Questions |

## Manual Assessment: Data Modeling

**Questions to Ask:**
1. Is there a documented data modeling approach (Star Schema, Data Vault, etc.)?
2. Are modeling standards enforced during code review?
3. Is there a data architecture diagram maintained?

**Evidence to Request:** Data modeling guidelines, Architecture diagrams, Code review checklist

**Scoring:** 5=Documented+enforced | 3=Some docs, not enforced | 1=No approach

**Manual Score (1-5):** _____

In [103]:
naming_df = session.sql(f"""
WITH db_names AS (
    SELECT NAME, CASE WHEN NAME RLIKE '.*_(DEV|UAT|TEST|PROD|STG)$' THEN 'ENV_SUFFIX'
        WHEN NAME RLIKE '^(DEV|UAT|TEST|PROD|STG)_.*' THEN 'ENV_PREFIX' ELSE 'NO_ENV_PATTERN' END as naming_pattern
    FROM {DATABASE}.{SCHEMA}.DATABASE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL),
wh_names AS (
    SELECT NAME, CASE WHEN NAME RLIKE '^WH_.*' OR NAME RLIKE '.*_WH$' THEN 'WH_CONVENTION'
        WHEN NAME RLIKE '.*(ETL|BI|DS|REPORT|LOAD).*' THEN 'WORKLOAD_BASED' ELSE 'NO_PATTERN' END as naming_pattern
    FROM {DATABASE}.{SCHEMA}.WAREHOUSE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL)
SELECT 'Databases' as object_type, naming_pattern, COUNT(*) as count FROM db_names GROUP BY 1, 2
UNION ALL SELECT 'Warehouses', naming_pattern, COUNT(*) FROM wh_names GROUP BY 1, 2 ORDER BY 1, 3 DESC
""").to_pandas()
print("=== Naming Convention Analysis ===")
display(naming_df)
db_pct = (naming_df[(naming_df['OBJECT_TYPE']=='Databases')&(naming_df['NAMING_PATTERN']!='NO_ENV_PATTERN')]['COUNT'].sum() / naming_df[naming_df['OBJECT_TYPE']=='Databases']['COUNT'].sum() * 100) if naming_df[naming_df['OBJECT_TYPE']=='Databases']['COUNT'].sum() > 0 else 0
wh_pct = (naming_df[(naming_df['OBJECT_TYPE']=='Warehouses')&(naming_df['NAMING_PATTERN']!='NO_PATTERN')]['COUNT'].sum() / naming_df[naming_df['OBJECT_TYPE']=='Warehouses']['COUNT'].sum() * 100) if naming_df[naming_df['OBJECT_TYPE']=='Warehouses']['COUNT'].sum() > 0 else 0
avg_pct = (db_pct + wh_pct) / 2
naming_score = 5 if avg_pct >= 80 else 4 if avg_pct >= 60 else 3 if avg_pct >= 40 else 2 if avg_pct >= 20 else 1
print(f"\nDatabases with pattern: {db_pct:.1f}% | Warehouses: {wh_pct:.1f}%")
print(f"📊 Naming Conventions Score: {naming_score}/5")

=== Naming Convention Analysis ===


,OBJECT_TYPE,NAMING_PATTERN,COUNT
0,Databases,NO_ENV_PATTERN,287
1,Databases,ENV_SUFFIX,19
2,Warehouses,WH_CONVENTION,156
3,Warehouses,NO_PATTERN,1



Databases with pattern: 6.2% | Warehouses: 99.4%
📊 Naming Conventions Score: 3/5


In [104]:
env_df = session.sql(f"""
SELECT CASE WHEN NAME ILIKE '%DEV%' THEN 'DEV' WHEN NAME ILIKE '%UAT%' OR NAME ILIKE '%TEST%' THEN 'TEST/UAT'
    WHEN NAME ILIKE '%STG%' THEN 'STAGING' WHEN NAME ILIKE '%PROD%' THEN 'PROD' ELSE 'UNCLASSIFIED' END as environment,
    COUNT(*) as db_count
FROM {DATABASE}.{SCHEMA}.DATABASE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL GROUP BY 1 ORDER BY 2 DESC
""").to_pandas()
print("=== Environment Strategy ===")
display(env_df)
envs = set(env_df[env_df['ENVIRONMENT'] != 'UNCLASSIFIED']['ENVIRONMENT'].tolist())
env_score = 5 if 'DEV' in envs and ('TEST/UAT' in envs or 'STAGING' in envs) and 'PROD' in envs else 4 if 'PROD' in envs and len(envs) > 1 else 3 if 'PROD' in envs else 2 if len(envs) > 0 else 1
print(f"📊 Environment Strategy Score: {env_score}/5")

=== Environment Strategy ===


,ENVIRONMENT,DB_COUNT
0,UNCLASSIFIED,271
1,PROD,16
2,DEV,14
3,TEST/UAT,3
4,STAGING,2


📊 Environment Strategy Score: 5/5


In [105]:
cap_df = session.sql(f"SELECT COUNT(*) as rm_count FROM {DATABASE}.{SCHEMA}.RESOURCE_MONITOR_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL").to_pandas()
rm_count = cap_df['RM_COUNT'].iloc[0]
print(f"=== Capacity Planning ===\nResource Monitors: {rm_count}")
capacity_ddm_score = 5 if rm_count >= 3 else 4 if rm_count >= 2 else 3 if rm_count >= 1 else 1
print(f"📊 Capacity Planning (DDM) Score: {capacity_ddm_score}/5")

=== Capacity Planning ===
Resource Monitors: 0
📊 Capacity Planning (DDM) Score: 1/5


## Manual Assessment: Capacity Planning

**Questions:**
1. Capacity planning before projects?
2. Quarterly growth forecasts?
3. Warehouse right-sizing process?

**Scoring:** 5=Regular reviews+forecasts | 3=Ad-hoc | 1=None

**Manual Score (1-5):** _____

In [106]:
data_modeling_score = 3; capacity_manual_score = 3  # Update from manual assessment
print("=" * 60 + "\nKEY TOPIC 1: PREPARE - Summary\n" + "=" * 60)
print(f"  Data Modeling (Manual):      {data_modeling_score}/5\n  Naming Conventions (DDM):    {naming_score}/5")
print(f"  Environment Strategy (DDM):  {env_score}/5\n  Capacity Planning (DDM):     {capacity_ddm_score}/5\n  Capacity Planning (Manual):  {capacity_manual_score}/5")
prepare_score = round((data_modeling_score + naming_score + env_score + capacity_ddm_score + capacity_manual_score) / 5, 1)
print(f"\n📊 PREPARE TOPIC SCORE: {prepare_score}/5")

KEY TOPIC 1: PREPARE - Summary
  Data Modeling (Manual):      3/5
  Naming Conventions (DDM):    3/5
  Environment Strategy (DDM):  5/5
  Capacity Planning (DDM):     1/5
  Capacity Planning (Manual):  3/5

📊 PREPARE TOPIC SCORE: 3.0/5


---
# Key Topic 2: IMPLEMENT

**Deploy with consistency and repeatability.**

| Sub-Topic | DDM Assessable | Assessment Method |
|-----------|----------------|-------------------|
| Infrastructure as Code | Yes | SESSION_ETL_V client patterns |
| CI/CD Pipelines | Partial | STAGE_ETL_V + Manual |
| Version Control | Yes | STAGE_ETL_V Git repos |
| Schema Migration | Partial | SESSION_ETL_V + Manual |

In [107]:
iac_df = session.sql("SELECT CLIENT_TOOL, IS_IAC, COUNT(*) as op_count, COUNT(DISTINCT USER_NAME) as users FROM cache_ddl_operations GROUP BY 1, 2 ORDER BY 3 DESC").to_pandas()
print("=== Infrastructure as Code ===")
display(iac_df)
total_ops = iac_df['OP_COUNT'].sum(); iac_ops = iac_df[iac_df['IS_IAC'] == True]['OP_COUNT'].sum()
iac_pct = (iac_ops / total_ops * 100) if total_ops > 0 else 0
iac_score = 5 if iac_pct >= 80 else 4 if iac_pct >= 60 else 3 if iac_pct >= 40 else 2 if iac_pct >= 20 else 1
print(f"\nIaC Adoption: {iac_pct:.1f}%\n📊 IaC Score: {iac_score}/5")

=== Infrastructure as Code ===


,CLIENT_TOOL,IS_IAC,OP_COUNT,USERS
0,Other/Manual,False,13432559,232
1,ODBC,False,5145995,476
2,JDBC,False,4125050,28
3,Python SDK,False,830695,57



IaC Adoption: 0.0%
📊 IaC Score: 1/5


In [108]:
git_df = session.sql(f"SELECT COUNT(CASE WHEN DELETED_ON IS NULL THEN 1 END) as active FROM {DATABASE}.{SCHEMA}.STAGE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND GIT_API_INTEGRATION IS NOT NULL").to_pandas()
active_repos = git_df['ACTIVE'].iloc[0]
print(f"=== Version Control ===\nActive Git repos: {active_repos}")
git_score = 5 if active_repos >= 5 else 4 if active_repos >= 3 else 3 if active_repos >= 1 else 1
print(f"📊 Version Control Score: {git_score}/5")

=== Version Control ===
Active Git repos: 48
📊 Version Control Score: 5/5


## Manual Assessment: CI/CD Pipelines

**Questions:**
1. Automated deployment pipelines?
2. Production approvals?
3. Automated testing?

**Scoring:** 5=Full automation+approvals | 3=Some automation | 1=All manual

**Manual Score (1-5):** _____

## Manual Assessment: Schema Migration

**Questions:**
1. Migration scripts tracked?
2. Rollback procedure?
3. Tested in lower envs?

**Scoring:** 5=Versioned+rollback | 3=Some tracking | 1=Ad-hoc

**Manual Score (1-5):** _____

In [109]:
cicd_score = 3; schema_migration_score = 3  # Update from manual
print("=" * 60 + "\nKEY TOPIC 2: IMPLEMENT - Summary\n" + "=" * 60)
print(f"  IaC (DDM): {iac_score}/5\n  CI/CD (Manual): {cicd_score}/5\n  Version Control (DDM): {git_score}/5\n  Schema Migration (Manual): {schema_migration_score}/5")
implement_score = round((iac_score + cicd_score + git_score + schema_migration_score) / 4, 1)
print(f"\n📊 IMPLEMENT TOPIC SCORE: {implement_score}/5")

KEY TOPIC 2: IMPLEMENT - Summary
  IaC (DDM): 1/5
  CI/CD (Manual): 3/5
  Version Control (DDM): 5/5
  Schema Migration (Manual): 3/5

📊 IMPLEMENT TOPIC SCORE: 3.0/5


---
# Key Topic 3: COLLABORATE

**Enable team effectiveness.**

| Sub-Topic | DDM Assessable | Assessment Method |
|-----------|----------------|-------------------|
| Documentation | No | Manual Questions |
| Standards | No | Manual Questions |
| Cross-Team Communication | Partial | TAG_ETL_V + Manual |

## Manual Assessment: Documentation

**Questions:**
1. Data dictionaries maintained?
2. Lineage documentation?
3. Runbooks for incidents?
4. Architecture diagrams?

**Scoring:** 5=Comprehensive+current | 3=Basic/outdated | 1=None

**Manual Score (1-5):** _____

## Manual Assessment: Standards

**Questions:**
1. SQL coding standards published?
2. Code review process?
3. Production approval workflows?
4. Testing standards?

**Scoring:** 5=Published+enforced | 3=Exists/inconsistent | 1=None

**Manual Score (1-5):** _____

In [110]:
tags_df = session.sql(f"SELECT COUNT(DISTINCT ID) as total, COUNT(CASE WHEN UPPER(NAME) LIKE '%OWNER%' OR UPPER(NAME) LIKE '%TEAM%' OR UPPER(NAME) LIKE '%COST%CENTER%' THEN ID END) as ownership FROM {DATABASE}.{SCHEMA}.TAG_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL").to_pandas()
ownership_tags = tags_df['OWNERSHIP'].iloc[0]
print(f"=== Ownership Tagging ===\nOwnership-related tags: {ownership_tags}")
tags_score = 5 if ownership_tags >= 3 else 4 if ownership_tags >= 2 else 3 if ownership_tags >= 1 else 2 if tags_df['TOTAL'].iloc[0] >= 1 else 1
print(f"📊 Ownership Tagging Score: {tags_score}/5")

=== Ownership Tagging ===
Ownership-related tags: 1
📊 Ownership Tagging Score: 3/5


## Manual Assessment: Cross-Team Communication

**Questions:**
1. Clear data asset ownership?
2. Communication channels between teams?
3. Cross-team data request process?

**Scoring:** 5=Clear ownership+channels+process | 3=Informal | 1=None

**Manual Score (1-5):** _____

In [111]:
documentation_score = 3; standards_score = 3; communication_score = 3  # Update from manual
print("=" * 60 + "\nKEY TOPIC 3: COLLABORATE - Summary\n" + "=" * 60)
print(f"  Documentation (Manual): {documentation_score}/5\n  Standards (Manual): {standards_score}/5\n  Ownership Tags (DDM): {tags_score}/5\n  Cross-Team Comm (Manual): {communication_score}/5")
collaborate_score = round((documentation_score + standards_score + tags_score + communication_score) / 4, 1)
print(f"\n�� COLLABORATE TOPIC SCORE: {collaborate_score}/5")

KEY TOPIC 3: COLLABORATE - Summary
  Documentation (Manual): 3/5
  Standards (Manual): 3/5
  Ownership Tags (DDM): 3/5
  Cross-Team Comm (Manual): 3/5

�� COLLABORATE TOPIC SCORE: 3.0/5


---
# Key Topic 4: AUTOMATE

**Reduce manual toil and human error.**

| Sub-Topic | DDM Assessable | Assessment Method |
|-----------|----------------|-------------------|
| Task Scheduling | Yes | USER_TASK_ETL_V, STREAM_ETL_V, TABLE_ETL_V |
| Alerting | Yes | ALERT_ETL_V |
| Self-Healing | Partial | USER_TASK_ETL_V + Manual |

In [112]:
auto_df = session.sql(f"""
SELECT 'Tasks' as feature, COUNT(*) as count FROM {DATABASE}.{SCHEMA}.USER_TASK_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
UNION ALL SELECT 'Streams', COUNT(*) FROM {DATABASE}.{SCHEMA}.STREAM_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
UNION ALL SELECT 'Dynamic Tables', COUNT(*) FROM {DATABASE}.{SCHEMA}.TABLE_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND KIND_ID = 13
""").to_pandas()
print("=== Automation Features ===")
display(auto_df)
auto_total = auto_df['COUNT'].sum()
scheduling_score = 5 if auto_total >= 50 else 4 if auto_total >= 20 else 3 if auto_total >= 5 else 2 if auto_total >= 1 else 1
print(f"\nTotal: {auto_total}\n📊 Task Scheduling Score: {scheduling_score}/5")

=== Automation Features ===


,FEATURE,COUNT
0,Tasks,5527
1,Streams,2779
2,Dynamic Tables,1060



Total: 9366
📊 Task Scheduling Score: 5/5


In [115]:
alerts_df = session.sql(f"SELECT COUNT(*) as ACTIVE FROM {DATABASE}.{SCHEMA}.ALERT_ETL_V WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL").to_pandas()
active_alerts = alerts_df['ACTIVE'].iloc[0]
print(f"=== Alerting ===\nActive alerts: {active_alerts}")
alerts_score = 5 if active_alerts >= 10 else 4 if active_alerts >= 5 else 3 if active_alerts >= 2 else 2 if active_alerts >= 1 else 1
print(f"📊 Alerting Score: {alerts_score}/5")

=== Alerting ===
Active alerts: 3
📊 Alerting Score: 3/5


## Manual Assessment: Self-Healing

**Questions:** 
1. Retry logic for transient failures?
2. Standardized error handling?
3. ALLOW_OVERLAPPING_EXECUTION configured?
4. USER_TASK_TIMEOUT_MS set?

**Scoring:** 5=Comprehensive retry+error handling | 3=Some handling | 1=No retry logic

**Manual Score (1-5):** _____

In [116]:
selfhealing_score = 3  # Update from manual
print("=" * 60 + "\nKEY TOPIC 4: AUTOMATE - Summary\n" + "=" * 60)
print(f"  Task Scheduling (DDM): {scheduling_score}/5\n  Alerting (DDM): {alerts_score}/5\n  Self-Healing (Manual): {selfhealing_score}/5")
automate_score = round((scheduling_score + alerts_score + selfhealing_score) / 3, 1)
print(f"\n📊 AUTOMATE TOPIC SCORE: {automate_score}/5")

KEY TOPIC 4: AUTOMATE - Summary
  Task Scheduling (DDM): 5/5
  Alerting (DDM): 3/5
  Self-Healing (Manual): 3/5

📊 AUTOMATE TOPIC SCORE: 3.7/5


---
# Overall Assessment Summary

In [117]:
print("=" * 70 + "\nOPERATIONAL EXCELLENCE - OVERALL ASSESSMENT SUMMARY\n" + "=" * 70)
print(f"Target: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}\n" + "=" * 70)
topics = {"1. Prepare": prepare_score, "2. Implement": implement_score, "3. Collaborate": collaborate_score, "4. Automate": automate_score}
print("\n📊 Topic Scores (1-5 Scale):" + "\n" + "-" * 50)
for topic, score in topics.items():
    bar = "█" * int(score) + "░" * (5 - int(score))
    status = "✅" if score >= 4 else "⚠️" if score >= 3 else "❌"
    print(f"{status} {topic}: {bar} {score}/5")
overall_score = round(sum(topics.values()) / len(topics), 1)
print("\n" + "=" * 50 + f"\n🏆 OVERALL OPERATIONAL EXCELLENCE SCORE: {overall_score}/5\n" + "=" * 50)
print("Rating: " + ("EXCELLENT" if overall_score >= 4 else "GOOD" if overall_score >= 3 else "MODERATE" if overall_score >= 2 else "NEEDS IMPROVEMENT"))
low_scores = [k for k, v in topics.items() if v < 3]
if low_scores: print("\n🎯 Priority Areas: " + ", ".join(low_scores))

OPERATIONAL EXCELLENCE - OVERALL ASSESSMENT SUMMARY
Target: SNOWHOUSE_IMPORT.IE | Account ID: 100058

📊 Topic Scores (1-5 Scale):
--------------------------------------------------
⚠️ 1. Prepare: ███░░ 3.0/5
⚠️ 2. Implement: ███░░ 3.0/5
⚠️ 3. Collaborate: ███░░ 3.0/5
⚠️ 4. Automate: ███░░ 3.7/5

🏆 OVERALL OPERATIONAL EXCELLENCE SCORE: 3.2/5
Rating: GOOD


In [118]:
prompt = f"""You are a Snowflake WAF advisor. Based on these scores, provide:
1. EXECUTIVE SUMMARY (2-3 sentences)
2. CRITICAL FINDINGS (top 3-5 issues with priority)
3. QUICK WINS (3-5 actions for 1 week)

Pillar: Operational Excellence | Overall: {overall_score}/5
Topics: {topics}
Low (<3): {[k for k,v in topics.items() if v < 3]} | High (>=4): {[k for k,v in topics.items() if v >= 4]}

Be specific based on actual scores."""
result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', '{prompt.replace(chr(39), chr(39)+chr(39))}') as response").collect()
print("=" * 70 + "\n🤖 AI-GENERATED INSIGHTS\n" + "=" * 70 + "\n" + result[0]['RESPONSE'])

🤖 AI-GENERATED INSIGHTS
1. EXECUTIVE SUMMARY
The Snowflake WAF implementation shows moderate performance across all operational excellence areas, with an overall score of 3.2/5. While automation practices are relatively stronger (3.7/5), the consistent 3.0 scores across preparation, implementation, and collaboration indicate systematic gaps that need attention. There are no critically low areas (<3.0) but also no exceptional strengths (≥4.0), suggesting room for balanced improvement.

2. CRITICAL FINDINGS
Priority 1: Preparation Gaps (3.0/5)
- Insufficient documentation and operational readiness
- Potential gaps in incident response planning
- Incomplete security baseline definitions

Priority 2: Implementation Weaknesses (3.0/5)
- Inconsistent deployment processes
- Limited monitoring and alerting coverage
- Gaps in change management procedures

Priority 3: Collaboration Barriers (3.0/5)
- Suboptimal cross-team communication
- Unclear roles and responsibilities
- Limited knowledge sha

---
## Assessment Coverage Matrix

| Topic | Sub-Topic | DDM | Manual | Source |
|-------|-----------|-----|--------|--------|
| Prepare | Data Modeling | Partial | ✅ | TABLE_ETL_V |
| Prepare | Naming | ✅ |
- | DATABASE/WAREHOUSE_ETL_V |
| Prepare | Environment | ✅ |
- | DATABASE_ETL_V |
| Prepare | Capacity | Partial | ✅ | RESOURCE_MONITOR_ETL_V |
| Implement | IaC | ✅ |
- | SESSION_ETL_V |
| Implement | CI/CD | Partial | ✅ | STAGE_ETL_V |
| Implement | Version Control | ✅ |
- | STAGE_ETL_V |
| Implement | Schema Migration | Partial | ✅ | SESSION_ETL_V |
| Collaborate | Documentation | ❌ | ✅ | N/A |
| Collaborate | Standards | ❌ | ✅ | N/A |
| Collaborate | Cross-Team | Partial | ✅ | TAG_ETL_V |
| Automate | Scheduling | ✅ |
- | TASK/STREAM/TABLE_ETL_V |
| Automate | Alerting | ✅ |
- | ALERT_ETL_V |
| Automate | Self-Healing | Partial | ✅ | USER_TASK_ETL_V |

**Summary:** 6 Fully DDM | 5 Partial+Manual | 3 Manual Only